# S07 -- atranka pagal pavadinimą ir santrauką

**A kelias (be programavimo).** Pagal protokolo kriterijus 18 įrašų atrenkami „include / borderline / exclude“. Pagal nutylėjimą rodomi **iš anksto išsaugoti** DKM sprendimai (nemokama). Matysite, kuriuos įrašus sistema palieka žmogui (*borderline*) ir kaip sutaria du vertintojai (Cohen kappa). Redaguokite antrojo vertintojo sprendimus (2 langelis) ir žiūrėkite, kaip keičiasi kappa.

Pažymėjus `USE_LIVE`, tikras DKM atrenka pirmus kelis įrašus pagal JŪSŲ kriterijus.

**B kelias (programuotojams)** -- žemiau.

### Prieš pradedant

1. **Pirmas paleidimas.** Paleidus pirmą kodo langelį, Colab gali parodyti įspėjimą *„Notebook was not authored by Google“*. Tai normalu (sąsiuvinis atkeltas iš GitHub): spauskite **Run anyway**.
2. **Pirmas langelis** atsisiunčia kodą ir įdiegia bibliotekas (maždaug 1-2 min.). Paleiskite jį pirmą ir palaukite, kol pasirodys `Paruosta:`.
3. **Tikras DKM nebūtinas.** Jei norite jį išbandyti: kairėje Colab juostoje spauskite rakto piktogramą (*Secrets*), *Add new secret*, pavadinimas lygiai **`OPENAI_API_KEY`**, reikšmė -- jūsų OpenAI raktas, ir įjunkite *Notebook access*. Raktą rašykite tik čia, niekada į kodo langelį. Kreipiniai apmokami.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/rimraul/slr-automation.git"
REPO_DIR = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")
if "google.colab" in sys.modules and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
for req in ['examples/chapter/requirements-colab.txt']:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit("pip install nepavyko (zr. klaida aukščiau): " + req)
print("Paruosta:", os.getcwd())

In [ ]:
#@title 1. Pasirinkite režimą
USE_LIVE = False  #@param {type:"boolean"}
# Tik USE_LIVE: kiek pirmų įrašų atrinkti (po 3 viename pakete; kiekvienas paketas -- vienas apmokamas kreipinys)
N_LIVE = 3  #@param {type:"slider", min:3, max:9, step:3}

In [ ]:
#@title 2. Redaguokite (second_rater -- visada; kriterijai -- tik kai USE_LIVE)
# Antrojo vertintojo sprendimai: "include", "borderline" arba "exclude".
# Pakeiskite kelias reikšmes ir paleiskite 3 langelį iš naujo -- kappa pasikeis.
SECOND_RATER = {
    'rec-01': 'include',
    'rec-03': 'include',
    'rec-05': 'exclude',
    'rec-07': 'include',
    'rec-08': 'exclude',
    'rec-10': 'borderline',
    'rec-13': 'exclude',
    'rec-14': 'exclude',
    'rec-15': 'exclude',
    'rec-16': 'include',
}

# Kriterijai DKM (veikia tik su USE_LIVE; pagal nutylėjimą -- mokomasis protokolas)
POPULATION = "Studies about the conduct of systematic, scoping or rapid literature reviews."
CONCEPT = "Automation of one or more review steps using LLMs, machine learning or rule-based tools."
CONTEXT = "Academic or applied research; any discipline; English; 2015 onward."
INCLUSION = [
    "The work describes or evaluates an automated or semi-automated method for a distinct review step.",
    "Automation of the named step is the subject of the report, not an incidental mention.",
]
EXCLUSION = [
    "The work is an opinion piece or editorial with no method and no evaluation.",
    "The work is about primary research automation rather than literature-review automation.",
    "Full text is not in English.",
]

In [ ]:
#@title 3. Atrinkti ir palyginti vertintojus
import re
from examples.chapter import toolkit
from examples.chapter.s07_screening import solution as s7

records = toolkit.load_fixture("bib_records.json")
if USE_LIVE:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    criteria = toolkit.Criteria(population=POPULATION, concept=CONCEPT, context=CONTEXT,
                                inclusion_criteria=tuple(INCLUSION), exclusion_criteria=tuple(EXCLUSION))
    records = records[:N_LIVE]
    print(f"Kreipiniai į DKM (apmokami): {len(records) // 3} pakeit(-as/-ų), {len(records)} įrašai")
    decisions = s7.screen_batch(records, criteria, mode="live")
else:
    scripted = toolkit.scripted_llm(toolkit.SCRIPTED_RESPONSES["s07"]["batches"],
                                    key_fn=lambda m: ",".join(re.findall(r"\[(rec-\d+)\]", m[-1]["content"])))
    decisions = s7.screen_batch(records, toolkit.DEMO_CRITERIA, call_llm=scripted)

by_id = {r["id"]: r for r in records}
print(f"{'id':7} {'DKM':11} {'2 vertintojas':13} pavadinimas")
for d in decisions:
    print(f"{d.record_id:7} {d.decision:11} {SECOND_RATER.get(d.record_id, '-'):13} {by_id[d.record_id]['title'][:60]}")

part = s7.partition(decisions)
print(f"\nĮtraukta automatiškai: {len(part.auto_included)} | atmesta automatiškai: {len(part.auto_excluded)}"
      f" | ŽMOGAUS EILĖ (borderline): {len(part.human_queue)} {part.human_queue}")

irr = s7.inter_rater_reliability({d.record_id: d.decision for d in decisions}, SECOND_RATER)
print(f"\nDKM prieš 2 vertintoją: kappa={irr.cohens_kappa:.2f} ({irr.kappa_label}), "
      f"sutapimas {irr.pct_agreement:.0%}, {irr.n_pairs} porų")
ref = s7.inter_rater_reliability(toolkit.load_fixture("expected_screening.json"), SECOND_RATER)
print(f"Etaloninis sąrašas prieš 2 vertintoją: kappa={ref.cohens_kappa:.2f} ({ref.kappa_label})")
print("Prisiminkite: ‘borderline’ visada tikrina žmogus; DKM sprendimas -- ne galutinis.")

In [ ]:
#@title (nebūtina) Ką DKM gauna iš tikrųjų?
from examples.chapter import toolkit
req = toolkit.build_screening_request(toolkit.load_fixture("bib_records.json")[:1], toolkit.DEMO_CRITERIA, mode="offline")
for m in req["messages"]:
    print(f"[{m['role']}]\n{m['content']}\n")

---
## B kelias: parašykite kodą patys

Toliau esantis pratimas skirtas tiems, kas nori išmokti programuoti.

# S07 pratimas

```
S7 pratimas -- pavadinimo/santraukos atranka ir tarpvertintojų patikimumas.

Užduotis
--------
1. Įgyvendinkite ``screen_batch(records, criteria, *, batch_size, mode,
   call_llm)``:
   - patikrinkite, kad ``1 <= batch_size <= 3`` (kitaip ``ValueError``);
   - kiekvienam paketui sudarykite užklausą per
     ``toolkit.build_screening_request`` ir kvieskite ``call_llm(**request)``;
   - patikrinkite, kad grąžinta po vieną galiojantį sprendimą kiekvienam
     įrašui; klaidos atveju iškelkite ``ValueError`` (NIEKADA nerašykite
     „exclude" iš atsargumo).
2. Įgyvendinkite ``partition`` -- „borderline" visada į žmogaus eilę.
3. Įgyvendinkite ``inter_rater_reliability`` -- dvireikšmis sutraukimas,
   Cohen's kappa, jos juostos etiketė.
4. Atrinkite ``bib_records.json``, apskaičiuokite kappa prieš
   ``second_rater_decisions.json`` ir interpretuokite juostą.
5. Neprivaloma (live): pakartokite vieną paketą su realiu API raktu.

Paleidimas: ``python -m examples.chapter.s07_screening.exercise``
```

1. Paleiskite paruošimo langelį. 2. Užpildykite `TODO` funkcijas pratimo langelyje ir paleiskite jį. 3. Palyginkite su pavyzdiniu sprendimu.

In [ ]:

from __future__ import annotations

from typing import Literal

from pydantic import BaseModel

from examples.chapter import toolkit


class ScreenDecision(BaseModel):
    record_id: str
    decision: Literal["include", "borderline", "exclude"]
    reason: str = ""


class Partition(BaseModel):
    auto_included: list[str]
    auto_excluded: list[str]
    human_queue: list[str]


class IRRReport(BaseModel):
    n_pairs: int
    pct_agreement: float
    cohens_kappa: float
    kappa_label: str


def screen_batch(records, criteria, *, batch_size=3, mode="offline", call_llm=toolkit.call_llm):
    """TODO: paketinė atranka su tikrinimu."""
    raise NotImplementedError


def partition(decisions) -> Partition:
    """TODO: „borderline" -> žmogaus eilė."""
    raise NotImplementedError


def inter_rater_reliability(first: dict, second: dict) -> IRRReport:
    """TODO: dvireikšmis sutraukimas + Cohen's kappa."""
    raise NotImplementedError


def main() -> None:
    import re

    records = toolkit.load_fixture("bib_records.json")
    scripted = toolkit.scripted_llm(
        toolkit.SCRIPTED_RESPONSES["s07"]["batches"],
        key_fn=lambda m: ",".join(re.findall(r"\[(rec-\d+)\]", m[-1]["content"])),
    )
    print("S7 screening exercise")
    try:
        decisions = screen_batch(records, toolkit.DEMO_CRITERIA, call_llm=scripted)
    except NotImplementedError:
        print("screen_batch() is not implemented yet -- fill it in and re-run.")
        return
    print(f"got {len(decisions)} decisions")


main()

## Pavyzdinis sprendimas

Pažiūrėkite tik pabandę patys. Žemiau esantys testai tikrina **pavyzdinį sprendimą**, ne jūsų kodą.

In [ ]:
!cat examples/chapter/s07_screening/solution.py

In [ ]:
!python -m pytest examples/chapter/s07_screening -q

In [ ]:
# NEPRIVALOMA: tikri kreipiniai i DKM (sumoka uz naudojimą). Raktą laikykite
# Colab "Secrets" (rakto piktograma kairėje) pavadinimu OPENAI_API_KEY; niekada ne užrašykite kode.
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")